# ViSongSearch – vòng 2: nhiều LLM viết truy vấn + LLM đoán tên bài (GPU)

1. *Add Input* → dataset **visongsearch-bundle-r1** (bản 2, có thư mục `vendor`).
2. *Session options*: **Accelerator = GPU T4 x2**, **Internet = On**.
3. (Không bắt buộc) *Add-ons → Secrets*: thêm `HF_TOKEN` nếu có tài khoản Hugging Face, để chạy thêm Llama-3.1.
4. **Save Version → Save & Run All (Commit)**. Ước tính 2,5–4 giờ. Mỗi mô hình lỗi thì bỏ qua và chạy mô hình sau.
5. Xong: tab *Output* → tải **visongsearch_results_r2.zip** → gửi lại cho nhóm.

Mô hình viết truy vấn: Qwen2.5-7B-Instruct, Qwen3-8B, SeaLLMs-v3-7B-Chat, SEA-LION-v3-8B-IT, Arcee-VyLinh (3B).
Mô hình đoán tên bài (closed-book và RAG): Qwen2.5-7B, Qwen3-8B, SeaLLMs-v3-7B.

In [ ]:
import glob, os, shutil, zipfile, subprocess, sys, time, json
ROOT = "/kaggle/working/vss"
if not os.path.exists(ROOT):
    inits = glob.glob("/kaggle/input/**/visong/__init__.py", recursive=True)
    zips = glob.glob("/kaggle/input/**/visongsearch-bundle*.zip", recursive=True)
    if inits:
        shutil.copytree(os.path.dirname(os.path.dirname(inits[0])), ROOT)
    elif zips:
        zipfile.ZipFile(zips[0]).extractall(ROOT)
    else:
        raise SystemExit("Chưa thêm dataset visongsearch-bundle-r1 (Add Input).")
os.chdir(ROOT); os.makedirs("logs", exist_ok=True)
print(sorted(os.listdir(ROOT)))
print(open("work_full/results/data_version.json").read())

In [ ]:
# Files round 2 adds to the round-1 bundle (written by scripts/make_r2_notebook.py)
FILES = {
"visong/llmtools.py": "\"\"\"Helpers for the round-2 LLM experiments (scripts/llm_r2.py): no GPU or model code here, so\neverything below is tested on CPU (tests/test_llmtools.py).\n\n1. Several open LLMs write the Q4 paraphrases and Q5 descriptions with the paper's prompts\n   (visong.queries.PROMPT_Q4 / PROMPT_Q5), so the results no longer rest on one generator.\n   Every output goes through visong.queries.check_llm_query plus the length limits that the\n   prompts state; a rejected output is regenerated with a new seed (at most `attempts` times).\n2. Closed-book identification: the LLM is asked which song a query is about, and its guesses\n   (title - artist) are matched to clusters by normalised title.\n3. Retrieval-augmented identification: the LLM picks the right song among the top lexical\n   candidates (listwise reranking).\n\"\"\"\nfrom __future__ import annotations\n\nimport random\nimport re\nfrom collections import defaultdict\n\nfrom .corpus import composer_of, singers_of, vietnamese_ratio\nfrom .dedup import norm_title\nfrom .queries import PROMPT_Q4, PROMPT_Q5, check_llm_query, in_llm_sample, representatives, sample_key\nfrom .text import clean, remove_diacritics\n\n# Length limits stated in the prompts (syllables after visong.text.clean).\nLENGTH = {\"Q4\": (8, 20), \"Q5\": (4, 35)}\n\nPROMPT_CLOSED = \"\"\"Một người đang cố tìm một bài hát tiếng Việt và gõ vào ô tìm kiếm:\n\"{query}\"\nĐó là bài hát nào? Hãy liệt kê tối đa 5 bài có khả năng nhất, bài khả năng nhất trước,\nmỗi dòng một bài theo đúng dạng: Tên bài - Ca sĩ hoặc nhạc sĩ. Không giải thích.\"\"\"\n\nPROMPT_RAG = \"\"\"Một người đang cố tìm một bài hát tiếng Việt và gõ vào ô tìm kiếm:\n\"{query}\"\nDưới đây là các bài hát ứng viên do máy tìm kiếm trả về (số thứ tự, tên bài, nghệ sĩ, một đoạn lời):\n{candidates}\nBài nào là bài người đó đang tìm? Chọn tối đa 5 bài phù hợp nhất, bài phù hợp nhất trước.\nChỉ trả lời các số thứ tự, cách nhau bằng dấu phẩy, ví dụ: 3, 1, 7\"\"\"\n\n\n# ------------------------------------------------------------- items for generation\n\ndef good_fragment(text: str) -> bool:\n    \"\"\"Same rule as scripts/prep_llm_items.py: two lines worth paraphrasing.\"\"\"\n    syl = clean(text).split()\n    return len(syl) >= 10 and len(set(syl)) >= 0.6 * len(syl) and vietnamese_ratio(text) >= 0.3\n\n\ndef item_of(song, cluster: str, frag: str | None, seed: int = 5) -> dict:\n    lines = song.lines()\n    if frag is None:\n        cands = [i for i in range(len(lines) - 1) if good_fragment(lines[i] + \" \" + lines[i + 1])]\n        i = random.Random(f\"{seed}:{song.doc_id}\").choice(cands) if cands else 0\n        frag = \" / \".join(lines[i:i + 2])\n    return {\"doc_id\": song.doc_id, \"cluster\": cluster, \"split\": song.extra[\"split\"], \"title\": song.title,\n            \"composer\": composer_of(song), \"singers\": \", \".join(singers_of(song)[:3]), \"frag\": frag,\n            \"lyrics_head\": \" / \".join(lines[:16])}\n\n\ndef build_items(songs, cluster_of: dict[str, str], existing: list[dict], n_test: int, n_dev: int,\n                seed: int = 5) -> list[dict]:\n    \"\"\"The clusters of the existing LLM queries first (same song, same Q4 fragment, so every\n    generator can be compared on paired queries), then more clusters of the same split in the\n    order of the LLM sample key (a uniform random order that does not depend on the crawl), until\n    the split has n_test / n_dev items.\"\"\"\n    by_id = {s.doc_id: s for s in songs}\n    reps = representatives(songs, cluster_of)\n    items, seen = [], set()\n    frag_of = {}\n    for q in existing:\n        if q[\"type\"] == \"Q4\" and q.get(\"meta\", {}).get(\"source_fragment\"):\n            frag_of[q[\"doc_id\"]] = q[\"meta\"][\"source_fragment\"]\n    for q in sorted(existing, key=lambda q: q[\"qid\"]):\n        if q[\"cluster\"] in seen or q[\"doc_id\"] not in by_id:\n            continue\n        seen.add(q[\"cluster\"])\n        it = item_of(by_id[q[\"doc_id\"]], q[\"cluster\"], frag_of.get(q[\"doc_id\"]), seed)\n        it[\"origin\"] = \"existing\"\n        items.append(it)\n    want = {\"test\": n_test, \"dev\": n_dev}\n    have = defaultdict(int)\n    for it in items:\n        have[it[\"split\"]] += 1\n    pool = []\n    for c, d in reps.items():\n        s = by_id[d]\n        if c in seen or s.extra.get(\"split\") not in want:\n            continue\n        pool.append((sample_key(s.extra.get(\"cluster_key\", c)), c, s))\n    for _, c, s in sorted(pool, key=lambda x: (x[0], x[1])):\n        split = s.extra[\"split\"]\n        if have[split] >= want[split]:\n            continue\n        it = item_of(s, c, None, seed)\n        if not good_fragment(it[\"frag\"].replace(\" / \", \" \")):\n            continue\n        it[\"origin\"] = \"sample\" if in_llm_sample(s) else \"extended\"\n        items.append(it)\n        have[split] += 1\n    return items\n\n\ndef prompt_for(qtype: str, item: dict) -> str:\n    if qtype == \"Q4\":\n        return PROMPT_Q4.format(fragment=item[\"frag\"])\n    return PROMPT_Q5.format(title=item[\"title\"], composer=item[\"composer\"] or \"không rõ\",\n                            singers=item[\"singers\"] or \"không rõ\", lyrics=item[\"lyrics_head\"])\n\n\n# ------------------------------------------------------------- cleaning and filtering outputs\n\n_THINK = re.compile(r\"<think>.*?</think>\", re.S)\n_LABEL = re.compile(r\"^\\s*(?:câu(?: tìm kiếm)?|truy vấn|query|trả lời|đáp án)\\s*[:：-]\\s*\", re.I)\n\n\ndef first_answer(text: str) -> str:\n    \"\"\"The first non-empty line of a reply, without a <think> block, a label or quotes.\"\"\"\n    text = _THINK.sub(\" \", text or \"\")\n    for line in text.splitlines():\n        q = \" \\t\\\"'“”‘’«»*`-•\"\n        line = _LABEL.sub(\"\", line.strip().strip(q)).strip(q)\n        if line:\n            return line\n    return \"\"\n\n\ndef filter_output(qtype: str, raw: str, song, frag: str | None) -> tuple[str, bool, str]:\n    \"\"\"Clean a generated query and apply the paper's filter plus the prompt's length limit.\"\"\"\n    text = clean(first_answer(raw))\n    if not text:\n        return text, False, \"empty\"\n    ok, why = check_llm_query(qtype, text, song, frag if qtype == \"Q4\" else None)\n    if not ok:\n        return text, False, why\n    lo, hi = LENGTH[qtype]\n    n = len(text.split())\n    if n < lo:\n        return text, False, \"too_short\"\n    if n > hi:\n        return text, False, \"too_long\"\n    return text, True, \"ok\"\n\n\n# ------------------------------------------------------------- closed-book identification\n\n_NUM = re.compile(r\"^\\s*(?:\\d+\\s*[.)/:-]|[-*•])\\s*\")\n_SEP = re.compile(r\"\\s+[-–—|]\\s+|\\s*[–—|]\\s*\")\n\n\ndef parse_guesses(text: str, k: int = 5) -> list[tuple[str, str]]:\n    \"\"\"Lines 'Tên bài - Nghệ sĩ' -> [(title, artist)], at most k, duplicates dropped.\"\"\"\n    out, seen = [], set()\n    for line in _THINK.sub(\" \", text or \"\").splitlines():\n        line = _NUM.sub(\"\", line.strip()).strip(\" \\\"'“”*`\")\n        if not line:\n            continue\n        parts = _SEP.split(line, maxsplit=1)\n        title = parts[0].strip(\" \\\"'“”*`\")\n        artist = parts[1].strip(\" \\\"'“”*`().\") if len(parts) > 1 else \"\"\n        key = norm_title(title)\n        if not key or key in seen:\n            continue\n        seen.add(key)\n        out.append((title, artist))\n        if len(out) >= k:\n            break\n    return out\n\n\ndef _artist_tokens(text: str) -> set[str]:\n    return set(remove_diacritics(clean(text)).split())\n\n\nclass TitleMatcher:\n    \"\"\"Normalised title -> clusters. A guessed title can match several clusters (common titles\n    such as 'Mưa' or 'Em'); the artist, when given, puts the clusters whose credits share a word\n    with it first, and the remaining ties are scored by their expected rank.\"\"\"\n\n    def __init__(self, songs, cluster_of: dict[str, str]):\n        self.by_title: dict[str, set[str]] = defaultdict(set)\n        self.credits: dict[str, set[str]] = defaultdict(set)\n        for s in songs:\n            c = cluster_of.get(s.doc_id, s.doc_id)\n            t = norm_title(s.title)\n            if t:\n                self.by_title[t].add(c)\n            self.credits[c] |= _artist_tokens(\" \".join([s.artist or \"\", composer_of(s), \" \".join(singers_of(s))]))\n\n    def blocks(self, guesses: list[tuple[str, str]]) -> list[list[str]]:\n        \"\"\"Ranked blocks of tied clusters, earlier guesses first; a cluster appears once.\"\"\"\n        out, seen = [], set()\n        for title, artist in guesses:\n            cl = sorted(self.by_title.get(norm_title(title), set()) - seen)\n            if not cl:\n                continue\n            a = _artist_tokens(artist) - {\"va\", \"ft\", \"feat\"}\n            if a:\n                first = [c for c in cl if a & self.credits[c]]\n                rest = [c for c in cl if c not in first]\n                for b in (first, rest):\n                    if b:\n                        out.append(b)\n            else:\n                out.append(cl)\n            seen.update(cl)\n        return out\n\n\ndef expected_rr(blocks: list[list[str]], gold: str, depth: int = 10) -> tuple[float, float | None]:\n    \"\"\"Expected reciprocal rank (cut at depth) when ties inside a block are in random order,\n    and the expected rank (None if the gold cluster is not in any block).\"\"\"\n    pos = 0\n    for b in blocks:\n        if gold in b:\n            ranks = range(pos + 1, pos + len(b) + 1)\n            rr = sum(1.0 / r for r in ranks if r <= depth) / len(b)\n            return rr, (pos + 1 + pos + len(b)) / 2\n        pos += len(b)\n    return 0.0, None\n\n\n# ------------------------------------------------------------- retrieval-augmented identification\n\ndef snippet(song, query: str, max_syl: int = 30) -> str:\n    \"\"\"The two consecutive lyric lines that share most unmarked syllables with the query,\n    cut to max_syl syllables, like a search-result snippet.\"\"\"\n    lines = [ln for ln in song.lines() if ln.strip()]\n    if not lines:\n        return \"\"\n    q = set(remove_diacritics(clean(query)).split())\n    best, best_i = -1, 0\n    for i in range(len(lines)):\n        win = remove_diacritics(clean(\" \".join(lines[i:i + 2]))).split()\n        sc = len(q & set(win))\n        if sc > best:\n            best, best_i = sc, i\n    toks = clean(\" / \".join(lines[best_i:best_i + 2]).replace(\"/\", \" / \")).split()\n    return \" \".join(toks[:max_syl])\n\n\ndef top_clusters(run: list[tuple[str, float]], cluster_of: dict[str, str], n: int) -> list[tuple[str, str]]:\n    \"\"\"The first n distinct clusters of a ranked list, each with its first-ranked version.\"\"\"\n    out, seen = [], set()\n    for d, _ in run:\n        c = cluster_of.get(d, d)\n        if c in seen:\n            continue\n        seen.add(c)\n        out.append((c, d))\n        if len(out) >= n:\n            break\n    return out\n\n\ndef candidate_block(cands: list[tuple[str, str]], by_id: dict, query: str) -> str:\n    rows = []\n    for i, (_, d) in enumerate(cands, 1):\n        s = by_id[d]\n        artist = s.artist or \", \".join(singers_of(s)[:2]) or \"không rõ\"\n        rows.append(f\"{i}. {s.title} | {artist} | {snippet(s, query)}\")\n    return \"\\n\".join(rows)\n\n\ndef parse_choice(text: str, n: int, k: int = 5) -> list[int]:\n    \"\"\"Candidate numbers (1..n) in the order given, duplicates dropped, at most k (0-based).\"\"\"\n    out = []\n    for m in re.finditer(r\"\\d+\", _THINK.sub(\" \", text or \"\")):\n        v = int(m.group()) - 1\n        if 0 <= v < n and v not in out:\n            out.append(v)\n        if len(out) >= k:\n            break\n    return out\n\n\ndef rerank(run: list[tuple[str, float]], cands: list[tuple[str, str]], chosen: list[int]) -> list[tuple[str, float]]:\n    \"\"\"Chosen candidates first (in the LLM's order), then the first-stage list unchanged.\"\"\"\n    first = [cands[i][1] for i in chosen]\n    rest = [d for d, _ in run if d not in set(first)]\n    order = first + rest\n    return [(d, float(len(order) - i)) for i, d in enumerate(order)]\n",
"scripts/llm_r2.py": "\"\"\"Round 2 on a Kaggle GPU: several open LLMs write Q4/Q5 queries, and LLM baselines identify songs.\n\n    python scripts/llm_r2.py run --work work_full [--models qwen25-7b,qwen3-8b,...]\n                                 [--identify qwen25-7b,qwen3-8b] [--deadline-min 540]\n    python scripts/llm_r2.py run --work work_full --backend fake      # CPU smoke test\n\nFor every model in --models, one after the other (each 7-8B model is loaded in float16 over both\nT4 GPUs, then deleted with its download):\n  1. generate: Q4 paraphrases and Q5 descriptions for every item of <work>/llm_prep/items_r2.json\n     (built here on first use: the clusters of the existing LLM queries, then a uniform sample of\n     more clusters, see visong.llmtools.build_items), with the paper's prompts, sampled at\n     temperature 0.7 / top-p 0.9; an output that fails the paper's filter (check_llm_query) or the\n     prompt's length limit is regenerated with another seed, at most --attempts times. Queries go\n     to queries/llm_multi.jsonl (+ llm_multi_unmarked.jsonl), statistics to results/llm_multi_stats.json.\n  2. identify (models in --identify only), on the test queries of the clusters that have the\n     existing Q4/Q5, for all seven types:\n     closed: the LLM names up to five songs (title - artist), matched to clusters by title;\n     rag:    the LLM picks up to five of the top-20 clusters of a lexical first stage\n             (equal-weight RRF of restored-syllable-bigram, unmarked-bigram, char-3 and metadata\n             BM25); the picks go first, the rest of the first-stage list follows.\n     Results go to results/llm_identify.json and runs/llm_{closed,rag}_<model>.trec.\nEverything is saved after each model, and a rerun skips the models already done.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport gc\nimport json\nimport os\nimport random\nimport shutil\nimport sys\nimport time\nfrom collections import Counter, defaultdict\n\nimport numpy as np\n\nsys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), \"..\"))\nfrom visong import corpus as C                                                    # noqa: E402\nfrom visong import llmtools as L                                                  # noqa: E402\nfrom visong.diacritics import DiacriticRestorer                                   # noqa: E402\nfrom visong.evaluate import bootstrap_ci, holm, per_query, randomization_test, write_trec  # noqa: E402\nfrom visong.fusion import rrf                                                     # noqa: E402\nfrom visong.lexical import LexicalIndex                                           # noqa: E402\nfrom visong.queries import longest_common_run, unmarked_variants                  # noqa: E402\nfrom visong.text import clean, has_diacritics, remove_diacritics                  # noqa: E402\n\nMODELS = {\n    \"qwen25-7b\": \"Qwen/Qwen2.5-7B-Instruct\",\n    \"qwen3-8b\": \"Qwen/Qwen3-8B\",\n    \"seallm3-7b\": \"SeaLLMs/SeaLLMs-v3-7B-Chat\",\n    \"sealion3-8b\": \"aisingapore/Llama-SEA-LION-v3-8B-IT\",\n    \"vylinh-3b\": \"arcee-ai/Arcee-VyLinh\",\n    \"llama31-8b\": \"meta-llama/Llama-3.1-8B-Instruct\",          # gated: needs HF_TOKEN\n}\nDEFAULT_MODELS = \"qwen25-7b,qwen3-8b,seallm3-7b,sealion3-8b,vylinh-3b\"\nID_TYPES = (\"Q1\", \"Q2\", \"Q3\", \"Q4\", \"Q5\", \"Q4u\", \"Q5u\")\n\n\ndef log(msg):\n    print(time.strftime(\"[%H:%M:%S] \") + msg, flush=True)\n\n\ndef path(work, *parts):\n    p = os.path.join(work, *parts)\n    os.makedirs(os.path.dirname(p), exist_ok=True)\n    return p\n\n\ndef load_json(p, default):\n    return json.load(open(p, encoding=\"utf-8\")) if os.path.exists(p) else default\n\n\ndef save_json(p, obj):\n    tmp = p + \".tmp\"\n    json.dump(obj, open(tmp, \"w\", encoding=\"utf-8\"), ensure_ascii=False, indent=1)\n    os.replace(tmp, p)\n\n\n# ------------------------------------------------------------------ backends\n\nclass HFChat:\n    \"\"\"A chat model from the Hugging Face hub in float16 over all visible GPUs.\"\"\"\n\n    def __init__(self, repo: str, token: str | None = None, cache_dir: str | None = None):\n        import torch\n        from transformers import AutoModelForCausalLM, AutoTokenizer\n        self.torch = torch\n        self.repo, self.cache_dir = repo, cache_dir\n        self.tok = AutoTokenizer.from_pretrained(repo, token=token, cache_dir=cache_dir)\n        self.tok.padding_side = \"left\"\n        if self.tok.pad_token is None:\n            self.tok.pad_token = self.tok.eos_token\n        kw = dict(token=token, cache_dir=cache_dir, device_map=\"auto\")\n        try:\n            self.model = AutoModelForCausalLM.from_pretrained(repo, dtype=torch.float16, **kw)\n        except TypeError:                       # transformers < 4.56 names it torch_dtype\n            self.model = AutoModelForCausalLM.from_pretrained(repo, torch_dtype=torch.float16, **kw)\n        self.model.eval()\n        self.revision = getattr(self.model.config, \"_commit_hash\", None)\n\n    def _template(self, prompt: str) -> str:\n        msgs = [{\"role\": \"user\", \"content\": prompt}]\n        try:        # Qwen3: no thinking block; other templates ignore the flag\n            return self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)\n        except Exception:\n            return self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)\n\n    def chat(self, prompts: list[str], temperature: float, top_p: float, max_new_tokens: int, seed: int,\n             batch_size: int) -> list[str]:\n        torch = self.torch\n        texts = [self._template(p) for p in prompts]\n        order = sorted(range(len(texts)), key=lambda i: -len(texts[i]))\n        out: list[str] = [\"\"] * len(texts)\n        start, bs = 0, batch_size\n        while start < len(order):\n            idx = order[start:start + bs]\n            enc = self.tok([texts[i] for i in idx], return_tensors=\"pt\", padding=True,\n                           add_special_tokens=False).to(self.model.device)\n            torch.manual_seed(seed + start)\n            kw = dict(do_sample=True, temperature=temperature, top_p=top_p) if temperature > 0 else dict(do_sample=False)\n            # float16 on a T4 can overflow in some models: invalid logits are replaced rather than\n            # stopping the run, and a batch that still fails is retried greedily, then left empty\n            # (an empty output is rejected and counted as \"empty\" in the statistics).\n            kw.update(remove_invalid_values=True, renormalize_logits=True)\n            gen, oom = None, False\n            for attempt in range(2):\n                try:\n                    with torch.no_grad():\n                        gen = self.model.generate(**enc, max_new_tokens=max_new_tokens,\n                                                  pad_token_id=self.tok.pad_token_id, **kw)\n                    break\n                except torch.cuda.OutOfMemoryError:\n                    oom = True\n                    break\n                except RuntimeError as e:\n                    self.failed_batches = getattr(self, \"failed_batches\", 0) + 1\n                    log(f\"  generation error ({'greedy retry' if attempt == 0 else 'batch left empty'}): {e!r}\"[:300])\n                    kw = dict(do_sample=False, remove_invalid_values=True)\n            if oom:\n                del enc\n                gc.collect()\n                torch.cuda.empty_cache()\n                if bs == 1:\n                    raise RuntimeError(\"out of memory with batch size 1\")\n                bs = max(1, bs // 2)\n                log(f\"  out of memory: batch size {bs}\")\n                continue\n            if gen is None:\n                start += len(idx)\n                continue\n            new = gen[:, enc[\"input_ids\"].shape[1]:]\n            for j, i in enumerate(idx):\n                out[i] = self.tok.decode(new[j], skip_special_tokens=True)\n            start += len(idx)\n        return out\n\n    def close(self):\n        del self.model\n        gc.collect()\n        self.torch.cuda.empty_cache()\n        if self.cache_dir and os.path.isdir(self.cache_dir):\n            shutil.rmtree(self.cache_dir, ignore_errors=True)\n\n\nclass FakeChat:\n    \"\"\"CPU stand-in for tests: paraphrases by dropping words, guesses the first candidate.\"\"\"\n    revision = \"fake\"\n\n    def __init__(self, repo, *a, **k):\n        self.repo = repo\n\n    def chat(self, prompts, temperature, top_p, max_new_tokens, seed, batch_size):\n        rng = random.Random(seed)\n        out = []\n        for p in prompts:\n            if \"ứng viên\" in p:\n                out.append(\"2, 1\")\n            elif \"liệt kê tối đa 5 bài\" in p:\n                out.append(\"1. Mưa - Không rõ\\n2. Em\")\n            elif \"Đây là đoạn lời thật\" in p:\n                frag = p.split('\"')[1]\n                toks = clean(frag).split()\n                keep = [t for t in toks if rng.random() < 0.6] + [\"nhớ\", \"mang\", \"máng\", \"bài\", \"này\"]\n                out.append(\" \".join(keep[:18]))\n            else:\n                out.append(\"bài hát buồn về mưa và người yêu cũ ở thành phố xa xôi nghe hồi xưa\")\n        return out\n\n    def close(self):\n        pass\n\n\ndef backend(kind: str, repo: str, token: str | None, cache_dir: str | None):\n    return FakeChat(repo) if kind == \"fake\" else HFChat(repo, token, cache_dir)\n\n\n# ------------------------------------------------------------------ data\n\ndef load_work(work):\n    songs = C.load_corpus(os.path.join(work, \"corpus.jsonl\"))\n    cluster_of = json.load(open(os.path.join(work, \"clusters.json\")))\n    return songs, cluster_of\n\n\ndef read(work, name):\n    p = os.path.join(work, \"queries\", name)\n    return C.read_jsonl(p) if os.path.exists(p) else []\n\n\ndef get_items(work, songs, cluster_of, n_test, n_dev):\n    p = path(work, \"llm_prep\", \"items_r2.json\")\n    if os.path.exists(p):\n        return json.load(open(p, encoding=\"utf-8\"))\n    items = L.build_items(songs, cluster_of, read(work, \"llm.jsonl\"), n_test, n_dev)\n    save_json(p, items)\n    log(f\"items {len(items)} {Counter((i['split'], i['origin']) for i in items)}\")\n    return items\n\n\n# ------------------------------------------------------------------ 1. generation\n\ndef text_stats(rows, songs_by_id):\n    \"\"\"Surface statistics of the generated queries (TMG: how the generators differ).\"\"\"\n    if not rows:\n        return {}\n    lens = [len(r[\"text\"].split()) for r in rows]\n    toks = [t for r in rows for t in r[\"text\"].split()]\n    big = [tuple(r[\"text\"].split()[i:i + 2]) for r in rows for i in range(len(r[\"text\"].split()) - 1)]\n    runs = []\n    for r in rows:\n        lyr = remove_diacritics(clean(\" \".join(songs_by_id[r[\"doc_id\"]].lines()))).split()\n        runs.append(longest_common_run(remove_diacritics(r[\"text\"]).split(), lyr))\n    return {\"n\": len(rows), \"mean_syllables\": float(np.mean(lens)), \"median_syllables\": float(np.median(lens)),\n            \"distinct1\": len(set(toks)) / max(1, len(toks)), \"distinct2\": len(set(big)) / max(1, len(big)),\n            \"mean_longest_lyric_run\": float(np.mean(runs)),\n            \"marked_share\": float(np.mean([has_diacritics(r[\"text\"]) for r in rows]))}\n\n\ndef generate(be, tag, items, by_id, a):\n    rows, rejects, st = [], [], {}\n    for qtype in (\"Q4\", \"Q5\"):\n        pending = list(range(len(items)))\n        reasons, final, first_try = Counter(), Counter(), 0\n        for attempt in range(a.attempts):\n            if not pending:\n                break\n            prompts = [L.prompt_for(qtype, items[i]) for i in pending]\n            outs = be.chat(prompts, a.temperature, a.top_p, a.max_new_tokens, a.seed + 1000 * attempt + (qtype == \"Q5\"),\n                           a.batch)\n            nxt = []\n            for i, raw in zip(pending, outs):\n                it = items[i]\n                text, ok, why = L.filter_output(qtype, raw, by_id[it[\"doc_id\"]], it[\"frag\"])\n                reasons[why] += 1\n                if ok:\n                    first_try += attempt == 0\n                    rows.append({\"qid\": f\"{it['doc_id']}-{qtype}-{tag}\", \"type\": qtype, \"text\": text,\n                                 \"doc_id\": it[\"doc_id\"], \"cluster\": it[\"cluster\"], \"split\": it[\"split\"],\n                                 \"meta\": {\"source_fragment\": it[\"frag\"] if qtype == \"Q4\" else None, \"generator\": tag,\n                                          \"model\": be.repo, \"revision\": be.revision, \"attempt\": attempt,\n                                          \"origin\": it[\"origin\"], \"length\": len(text.split())}})\n                else:\n                    nxt.append(i)\n                    if attempt == a.attempts - 1:\n                        final[why] += 1\n                    rejects.append({\"generator\": tag, \"type\": qtype, \"doc_id\": it[\"doc_id\"], \"attempt\": attempt,\n                                    \"reason\": why, \"text\": text[:300]})\n            pending = nxt\n        st[qtype] = {\"items\": len(items), \"accepted\": len(items) - len(pending), \"accepted_first_try\": first_try,\n                     \"outputs_by_reason\": dict(reasons), \"dropped_by_last_reason\": dict(final)}\n        log(f\"  {tag} {qtype}: {st[qtype]['accepted']}/{len(items)} accepted ({first_try} first try) {dict(reasons)}\")\n    for qtype in (\"Q4\", \"Q5\"):\n        st[qtype][\"text\"] = text_stats([r for r in rows if r[\"type\"] == qtype], by_id)\n    return rows, rejects, st\n\n\n# ------------------------------------------------------------------ 2. identification\n\ndef id_queries(work, max_per_type: int | None, seed: int = 0):\n    \"\"\"Test queries of the clusters that have the existing Q4/Q5, all seven types.\"\"\"\n    llm = [q for q in read(work, \"llm.jsonl\") if q[\"split\"] == \"test\"]\n    clusters = sorted({q[\"cluster\"] for q in llm})\n    if max_per_type and len(clusters) > max_per_type:\n        clusters = sorted(random.Random(seed).sample(clusters, max_per_type))\n    keep = set(clusters)\n    qs = [q for q in read(work, \"rule.jsonl\") + llm + read(work, \"llm_unmarked.jsonl\")\n          if q[\"split\"] == \"test\" and q[\"cluster\"] in keep and q[\"type\"] in ID_TYPES]\n    return qs\n\n\ndef tune(index, dev, cluster_of, grid_k1=(0.3, 0.6, 0.9, 1.2, 1.5), grid_b=(0.3, 0.5, 0.75, 0.9)):\n    def m(run):\n        pq = per_query(run, dev, cluster_of)\n        return float(np.mean([v[\"rr10\"] for v in pq.values()])) if pq else 0.0\n    best = (-1.0, (0.9, 0.4))\n    for k1 in grid_k1:\n        for b in grid_b:\n            index.set_params(k1, b)\n            s = m({q[\"qid\"]: index.search(q[\"text\"]) for q in dev})\n            if s > best[0]:\n                best = (s, (k1, b))\n    index.set_params(*best[1])\n    return {\"k1\": best[1][0], \"b\": best[1][1], \"dev_mrr10\": best[0]}\n\n\ndef first_stage(work, songs, cluster_of, queries, seed=0):\n    \"\"\"Equal-weight RRF of four lexical lists (the lists of the paper's lexical fusion),\n    BM25 k1/b tuned on dev queries.\"\"\"\n    t0 = time.time()\n    rng = random.Random(seed)\n    dev = [q for q in read(work, \"rule.jsonl\") if q[\"split\"] == \"dev\"]\n    dev = rng.sample(dev, min(900, len(dev))) + [q for q in read(work, \"llm.jsonl\") + read(work, \"llm_unmarked.jsonl\")\n                                                 if q[\"split\"] == \"dev\"]\n    texts, ids = [s.lyrics for s in songs], [s.doc_id for s in songs]\n    meta = [\" \".join(x for x in (s.title, s.artist, s.extra.get(\"singers\", \"\"), s.genre) if x) for s in songs]\n    rest = DiacriticRestorer(texts)\n    restored = {q[\"qid\"]: rest.restore(q[\"text\"]) for q in queries + dev}\n    params, lists = {}, defaultdict(dict)\n    specs = [(\"restoreall_bm25_syl12\", texts, \"syl12\", True), (\"bm25_nodiac12\", texts, \"nodiac12\", False),\n             (\"bm25_char3\", texts, \"char3\", False), (\"bm25_meta\", meta, \"nodiac12\", False)]\n    for name, docs, an, use_rest in specs:\n        idx = LexicalIndex(docs, ids, an)\n        dq = [{**q, \"text\": restored[q[\"qid\"]]} for q in dev] if use_rest else dev\n        params[name] = tune(idx, dq, cluster_of)\n        for q in queries:\n            lists[name][q[\"qid\"]] = idx.search(restored[q[\"qid\"]] if use_rest else q[\"text\"], k=100)\n        log(f\"  first stage {name} {params[name]}\")\n        del idx\n        gc.collect()\n    run = {q[\"qid\"]: rrf([lists[n][q[\"qid\"]] for n, *_ in specs]) for q in queries}\n    params[\"seconds\"] = round(time.time() - t0, 1)\n    return run, params\n\n\ndef summarize_types(pq: dict, queries, key=\"rr10\"):\n    out = {}\n    for t in ID_TYPES:\n        qids = [q[\"qid\"] for q in queries if q[\"type\"] == t and q[\"qid\"] in pq]\n        if qids:\n            vals = np.array([pq[x][key] for x in qids], dtype=float)\n            lo, hi = bootstrap_ci(vals)\n            out[t] = {\"n\": len(qids), \"mrr10\": float(vals.mean()), \"ci95\": [lo, hi]}\n    return out\n\n\ndef identify_closed(be, tag, queries, matcher, a):\n    prompts = [L.PROMPT_CLOSED.format(query=q[\"text\"]) for q in queries]\n    outs = be.chat(prompts, 0.0, 1.0, 160, a.seed, a.batch_id)\n    pq, trec = {}, {}\n    rng = random.Random(a.seed)\n    for q, raw in zip(queries, outs):\n        guesses = L.parse_guesses(raw)\n        blocks = matcher.blocks(guesses)\n        rr, erank = L.expected_rr(blocks, q[\"cluster\"])\n        top5 = set(c for b in blocks[:5] for c in b)\n        pq[q[\"qid\"]] = {\"rr10\": rr, \"expected_rank\": erank, \"n_guesses\": len(guesses),\n                        \"n_matched\": sum(len(b) for b in blocks), \"lenient_hit5\": float(q[\"cluster\"] in top5),\n                        \"raw\": raw[:300]}\n        order = []\n        for b in blocks:\n            b = list(b)\n            rng.shuffle(b)\n            order += b\n        trec[q[\"qid\"]] = [(c, float(len(order) - i)) for i, c in enumerate(order)]\n    return pq, trec\n\n\ndef identify_rag(be, tag, queries, first, by_id, cluster_of, a):\n    cands = {q[\"qid\"]: L.top_clusters(first[q[\"qid\"]], cluster_of, a.rag_k) for q in queries}\n    prompts = [L.PROMPT_RAG.format(query=q[\"text\"], candidates=L.candidate_block(cands[q[\"qid\"]], by_id, q[\"text\"]))\n               for q in queries]\n    outs = be.chat(prompts, 0.0, 1.0, 40, a.seed, a.batch_rag)\n    run, extra = {}, {}\n    for q, raw in zip(queries, outs):\n        c = cands[q[\"qid\"]]\n        chosen = L.parse_choice(raw, len(c))\n        run[q[\"qid\"]] = L.rerank(first[q[\"qid\"]], c, chosen)\n        extra[q[\"qid\"]] = {\"chosen\": chosen, \"gold_in_candidates\": float(q[\"cluster\"] in {x for x, _ in c}),\n                           \"raw\": raw[:120]}\n    pq = per_query(run, queries, cluster_of)\n    for k, v in extra.items():\n        pq[k].update(v)\n    return pq, run\n\n\ndef significance(res, queries):\n    \"\"\"Paired randomization tests per type: each LLM run vs the first stage (RAG) and the RAG run\n    vs the closed-book run of the same model; Holm over all tests of the file.\"\"\"\n    p, diffs = {}, {}\n    fs = res[\"per_query\"].get(\"first_stage\", {})\n    for mode in (\"rag\", \"closed\"):\n        for tag, pq in res[\"per_query\"].get(mode, {}).items():\n            for t in ID_TYPES:\n                qids = [q[\"qid\"] for q in queries if q[\"type\"] == t and q[\"qid\"] in pq and q[\"qid\"] in fs]\n                if len(qids) < 5:\n                    continue\n                a_ = np.array([pq[x][\"rr10\"] for x in qids])\n                b_ = np.array([fs[x][\"rr10\"] for x in qids])\n                key = f\"{mode}_{tag}_vs_first_stage_{t}\"\n                p[key] = randomization_test(a_, b_)\n                diffs[key] = float(a_.mean() - b_.mean())\n    adj = holm(p) if p else {}\n    return {k: {\"diff\": diffs[k], \"p\": p[k], \"p_holm\": adj.get(k)} for k in p}\n\n\n# ------------------------------------------------------------------ main\n\ndef run(a):\n    t_start = time.time()\n    deadline = t_start + 60 * a.deadline_min\n    work = a.work\n    songs, cluster_of = load_work(work)\n    by_id = {s.doc_id: s for s in songs}\n    items = get_items(work, songs, cluster_of, a.test, a.dev)\n    token = os.environ.get(\"HF_TOKEN\") or None\n    tags = [t for t in a.models.split(\",\") if t]\n    id_tags = set(t for t in (a.identify or \"\").split(\",\") if t)\n    out_q = path(work, \"queries\", \"llm_multi.jsonl\")\n    out_rej = path(work, \"results\", \"llm_multi_rejects.jsonl\")\n    stats_p = path(work, \"results\", \"llm_multi_stats.json\")\n    id_p = path(work, \"results\", \"llm_identify.json\")\n    stats = load_json(stats_p, {\"params\": {}, \"models\": {}})\n    stats[\"params\"] = {\"temperature\": a.temperature, \"top_p\": a.top_p, \"max_new_tokens\": a.max_new_tokens,\n                       \"attempts\": a.attempts, \"seed\": a.seed, \"length_limits\": L.LENGTH, \"dtype\": \"float16\",\n                       \"items\": dict(Counter(f\"{i['split']}/{i['origin']}\" for i in items))}\n    rows = read(work, \"llm_multi.jsonl\")\n    res = load_json(id_p, {\"params\": {}, \"first_stage\": {}, \"closed\": {}, \"rag\": {}, \"per_query\": {}})\n\n    queries, first, matcher = [], None, None\n    if id_tags:\n        queries = id_queries(work, a.id_max)\n        log(f\"identification queries {Counter(q['type'] for q in queries)}\")\n        first, fs_params = first_stage(work, songs, cluster_of, queries)\n        pq = per_query(first, queries, cluster_of)\n        res[\"first_stage\"] = summarize_types(pq, queries)\n        res[\"per_query\"][\"first_stage\"] = {k: {\"rr10\": v[\"rr10\"]} for k, v in pq.items()}\n        res[\"params\"] = {\"first_stage\": fs_params, \"rag_k\": a.rag_k, \"decoding\": \"greedy\",\n                         \"closed_max_new_tokens\": 160, \"rag_max_new_tokens\": 40}\n        write_trec(path(work, \"runs\", \"llm_first_stage.trec\"), first, \"llm_first_stage\")\n        matcher = L.TitleMatcher(songs, cluster_of)\n        log(f\"first stage {res['first_stage']}\")\n        save_json(id_p, res)\n\n    for tag in tags:\n        need_gen = not stats[\"models\"].get(tag, {}).get(\"done\")\n        need_id = tag in id_tags and not all(tag in res.get(m, {}) for m in (\"closed\", \"rag\"))\n        if not (need_gen or need_id):\n            log(f\"{tag}: already done\")\n            continue\n        if time.time() > deadline:\n            log(f\"deadline reached before {tag}\")\n            break\n        repo = MODELS.get(tag, tag)\n        t0 = time.time()\n        try:\n            be = backend(a.backend, repo, token, os.path.join(a.cache, tag))\n        except Exception as e:                      # gated, missing, or does not fit\n            log(f\"{tag}: could not load {repo}: {e!r}\"[:600])\n            stats[\"models\"][tag] = {\"model\": repo, \"error\": repr(e)[:600]}\n            save_json(stats_p, stats)\n            continue\n        load_s = round(time.time() - t0, 1)\n        try:\n            if need_gen:\n                t1 = time.time()\n                new, rejects, st = generate(be, tag, items, by_id, a)\n                rows = [r for r in rows if r[\"meta\"].get(\"generator\") != tag] + new\n                C.write_jsonl(out_q, rows)\n                C.write_jsonl(path(work, \"queries\", \"llm_multi_unmarked.jsonl\"), unmarked_variants(rows))\n                with open(out_rej, \"a\", encoding=\"utf-8\") as f:\n                    for r in rejects:\n                        f.write(json.dumps(r, ensure_ascii=False) + \"\\n\")\n                stats[\"models\"][tag] = {\"model\": repo, \"revision\": be.revision, \"load_seconds\": load_s,\n                                        \"failed_batches\": getattr(be, \"failed_batches\", 0),\n                                        \"generate_seconds\": round(time.time() - t1, 1), **st, \"done\": True}\n                save_json(stats_p, stats)\n            if need_id and time.time() < deadline:\n                t1 = time.time()\n                pq, trec = identify_closed(be, tag, queries, matcher, a)\n                res[\"closed\"][tag] = summarize_types(pq, queries)\n                for t, v in res[\"closed\"][tag].items():\n                    v[\"lenient_hit5\"] = float(np.mean([pq[q[\"qid\"]][\"lenient_hit5\"] for q in queries if q[\"type\"] == t]))\n                res[\"per_query\"].setdefault(\"closed\", {})[tag] = pq\n                write_trec(path(work, \"runs\", f\"llm_closed_{tag}.trec\"),\n                           {k: [(d, s) for d, s in v] for k, v in trec.items()}, f\"llm_closed_{tag}\")\n                log(f\"  {tag} closed {round(time.time() - t1)}s {res['closed'][tag]}\")\n                save_json(id_p, res)\n                t1 = time.time()\n                pq, rrun = identify_rag(be, tag, queries, first, by_id, cluster_of, a)\n                res[\"rag\"][tag] = summarize_types(pq, queries)\n                for t, v in res[\"rag\"][tag].items():\n                    v[\"first_stage_recall_k\"] = float(np.mean([pq[q[\"qid\"]][\"gold_in_candidates\"]\n                                                               for q in queries if q[\"type\"] == t]))\n                res[\"per_query\"].setdefault(\"rag\", {})[tag] = pq\n                write_trec(path(work, \"runs\", f\"llm_rag_{tag}.trec\"), rrun, f\"llm_rag_{tag}\")\n                log(f\"  {tag} rag {round(time.time() - t1)}s {res['rag'][tag]}\")\n                res.setdefault(\"models\", {})[tag] = {\"model\": repo, \"revision\": be.revision}\n                save_json(id_p, res)\n        except Exception as e:\n            log(f\"{tag}: failed: {e!r}\"[:600])\n            stats[\"models\"].setdefault(tag, {})[\"error\"] = repr(e)[:600]\n            save_json(stats_p, stats)\n        finally:\n            be.close()\n            del be\n            gc.collect()\n    if queries:\n        res[\"significance\"] = significance(res, queries)\n        save_json(id_p, res)\n    stats[\"total_minutes\"] = round((time.time() - t_start) / 60, 1)\n    save_json(stats_p, stats)\n    log(f\"done in {stats['total_minutes']} min; {len(rows)} queries; \"\n        f\"{Counter((r['meta']['generator'], r['type']) for r in rows)}\")\n\n\ndef main(argv=None):\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument(\"cmd\", choices=[\"run\", \"items\"])\n    ap.add_argument(\"--work\", default=\"work_full\")\n    ap.add_argument(\"--models\", default=DEFAULT_MODELS)\n    ap.add_argument(\"--identify\", default=\"qwen25-7b,qwen3-8b,seallm3-7b\", help=\"models that also run the LLM baselines\")\n    ap.add_argument(\"--backend\", default=\"hf\", choices=[\"hf\", \"fake\"])\n    ap.add_argument(\"--cache\", default=\"/tmp/hf_r2\", help=\"download folder, deleted after each model\")\n    ap.add_argument(\"--test\", type=int, default=600, help=\"test items for generation\")\n    ap.add_argument(\"--dev\", type=int, default=150, help=\"dev items for generation\")\n    ap.add_argument(\"--attempts\", type=int, default=3)\n    ap.add_argument(\"--temperature\", type=float, default=0.7)\n    ap.add_argument(\"--top-p\", type=float, default=0.9)\n    ap.add_argument(\"--max-new-tokens\", type=int, default=96)\n    ap.add_argument(\"--batch\", type=int, default=32)\n    ap.add_argument(\"--batch-id\", type=int, default=32)\n    ap.add_argument(\"--batch-rag\", type=int, default=8)\n    ap.add_argument(\"--rag-k\", type=int, default=20)\n    ap.add_argument(\"--id-max\", type=int, default=None, help=\"at most this many clusters for identification\")\n    ap.add_argument(\"--seed\", type=int, default=7)\n    ap.add_argument(\"--deadline-min\", type=float, default=540)\n    a = ap.parse_args(argv)\n    if a.cmd == \"items\":\n        songs, cluster_of = load_work(a.work)\n        get_items(a.work, songs, cluster_of, a.test, a.dev)\n        return\n    run(a)\n\n\nif __name__ == \"__main__\":\n    main()\n",
"tests/test_llmtools.py": "from visong import llmtools as L\nfrom visong.corpus import Song\n\n\ndef song(doc_id, title, artist=\"\", lyrics=\"\", split=\"test\", singers=\"\"):\n    return Song(doc_id=doc_id, url=f\"https://x/{doc_id}\", source=\"x\", title=title, artist=artist, lyrics=lyrics,\n                extra={\"split\": split, \"cluster_key\": doc_id, \"singers\": singers})\n\n\nLYR = (\"Chiều nay mưa rơi trên phố vắng em đi\\nAnh đứng lặng nhìn theo bóng dáng ai xa\\n\"\n       \"Ngày xưa ta hẹn thề bên nhau mãi mãi\\nGiờ chỉ còn mình anh với cơn mưa chiều\")\n\n\ndef test_first_answer_strips_labels_quotes_and_think():\n    assert L.first_answer('<think>hmm</think>\\n\\n\"Câu tìm kiếm: mưa chiều phố vắng\"') == \"mưa chiều phố vắng\"\n    assert L.first_answer(\"  \\n- bài hát về mưa\\nthêm dòng\") == \"bài hát về mưa\"\n    assert L.first_answer(\"\") == \"\"\n\n\ndef test_filter_output_length_and_copy():\n    s = song(\"a1\", \"Mưa Chiều\", lyrics=LYR)\n    frag = \"Chiều nay mưa rơi trên phố vắng em đi / Anh đứng lặng nhìn theo bóng dáng ai xa\"\n    text, ok, why = L.filter_output(\"Q4\", \"buổi chiều trời đổ mưa con phố không người em bước đi xa\", s, frag)\n    assert ok and why == \"ok\", why\n    _, ok, why = L.filter_output(\"Q4\", \"chiều nay mưa rơi trên phố vắng em đi mất rồi\", s, frag)\n    assert not ok and why.startswith(\"copied\") or why == \"too_close\"\n    _, ok, why = L.filter_output(\"Q4\", \"trời buồn quá đi thôi\", s, frag)\n    assert not ok and why == \"too_short\"\n    long = \" \".join([\"mưa\"] * 3 + [\"ngày xưa có một người con gái bước đi trong chiều mưa lạnh lẽo buồn hiu hắt mãi không quay về nữa\"])\n    _, ok, why = L.filter_output(\"Q4\", long, s, frag)\n    assert not ok and why == \"too_long\"\n\n\ndef test_parse_guesses():\n    g = L.parse_guesses(\"1. Mưa Chiều - Lam Phương\\n2) Em Đi – Trịnh Công Sơn\\n3. Mưa Chiều - X\\n4. Diễm Xưa\")\n    assert g == [(\"Mưa Chiều\", \"Lam Phương\"), (\"Em Đi\", \"Trịnh Công Sơn\"), (\"Diễm Xưa\", \"\")]\n\n\ndef test_title_matcher_blocks_and_expected_rr():\n    songs = [song(\"a\", \"Mưa\", \"Lam Phương\"), song(\"b\", \"Mưa\", \"Trịnh Công Sơn\"), song(\"c\", \"Mưa (Remix)\", \"Ai Đó\"),\n             song(\"d\", \"Em Đi\", \"Trịnh Công Sơn\")]\n    cluster_of = {\"a\": \"a\", \"b\": \"b\", \"c\": \"c\", \"d\": \"d\"}\n    m = L.TitleMatcher(songs, cluster_of)\n    blocks = m.blocks([(\"Mưa\", \"Trịnh Công Sơn\"), (\"Em Đi\", \"\")])\n    assert blocks[0] == [\"b\"] and set(blocks[1]) == {\"a\", \"c\"} and blocks[2] == [\"d\"]\n    assert L.expected_rr(blocks, \"b\")[0] == 1.0\n    rr, er = L.expected_rr(blocks, \"a\")\n    assert abs(rr - (1 / 2 + 1 / 3) / 2) < 1e-9 and er == 2.5\n    assert L.expected_rr(blocks, \"zzz\") == (0.0, None)\n    assert L.expected_rr([[\"x\"] * 12, [\"g\"]], \"g\")[0] == 0.0\n\n\ndef test_choice_and_rerank():\n    assert L.parse_choice(\"3, 1, 3, 9, 25\", n=10) == [2, 0, 8]\n    cands = [(\"c1\", \"d1\"), (\"c2\", \"d2\"), (\"c3\", \"d3\")]\n    run = [(\"d1\", 3.0), (\"d2\", 2.0), (\"d3\", 1.0), (\"d4\", 0.5)]\n    out = [d for d, _ in L.rerank(run, cands, [2])]\n    assert out == [\"d3\", \"d1\", \"d2\", \"d4\"]\n\n\ndef test_snippet_and_top_clusters():\n    s = song(\"a\", \"Mưa Chiều\", lyrics=LYR)\n    sn = L.snippet(s, \"hen the ben nhau mai mai\")\n    assert \"hẹn thề\" in sn\n    assert L.top_clusters([(\"a\", 1), (\"a2\", 0.9), (\"b\", 0.8)], {\"a\": \"A\", \"a2\": \"A\", \"b\": \"B\"}, 5) == [(\"A\", \"a\"), (\"B\", \"b\")]\n\n\ndef test_build_items_keeps_existing_then_samples():\n    songs = [song(f\"s{i}\", f\"Bài {i}\", lyrics=LYR, split=\"test\" if i % 2 else \"dev\") for i in range(10)]\n    cluster_of = {s.doc_id: s.doc_id for s in songs}\n    existing = [{\"qid\": \"s1-Q4-0\", \"type\": \"Q4\", \"doc_id\": \"s1\", \"cluster\": \"s1\", \"split\": \"test\",\n                 \"meta\": {\"source_fragment\": \"frag cũ\"}}]\n    items = L.build_items(songs, cluster_of, existing, n_test=3, n_dev=2)\n    assert items[0][\"doc_id\"] == \"s1\" and items[0][\"frag\"] == \"frag cũ\" and items[0][\"origin\"] == \"existing\"\n    assert sum(i[\"split\"] == \"test\" for i in items) == 3 and sum(i[\"split\"] == \"dev\" for i in items) == 2\n    assert len({i[\"cluster\"] for i in items}) == len(items)\n    assert \"Bài 1\" not in L.prompt_for(\"Q4\", items[0]) and \"Bài 1\" in L.prompt_for(\"Q5\", items[0])\n"
}
for rel, text in FILES.items():
    os.makedirs(os.path.dirname(os.path.join(ROOT, rel)), exist_ok=True)
    open(os.path.join(ROOT, rel), 'w', encoding='utf-8').write(text)
print('wrote', list(FILES))

In [ ]:
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "accelerate", "pyvi"], capture_output=True, text=True)
print(r.stdout[-300:], r.stderr[-800:])
sys.path.insert(0, os.path.join(ROOT, "vendor")); sys.path.insert(0, ROOT)
os.environ["PYTHONPATH"] = os.path.join(ROOT, "vendor") + os.pathsep + ROOT + os.pathsep + os.environ.get("PYTHONPATH", "")
os.environ["TOKENIZERS_PARALLELISM"] = "false"; os.environ["PYTHONWARNINGS"] = "ignore"
import datasketch, torch, transformers, accelerate
import urllib.request
try:
    urllib.request.urlopen("https://huggingface.co", timeout=20)
except Exception as e:
    raise SystemExit(f"Không vào được huggingface.co ({e}): bật Session options → Internet = On rồi chạy lại")
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN: có")
except Exception:
    print("HF_TOKEN: không có (bỏ qua các mô hình cần đăng nhập)")
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] or "KHÔNG CÓ GPU - bật Accelerator")

In [ ]:
def run(name, cmd):
    """Stream a command's output into the notebook and logs/<name>.log."""
    t0 = time.time(); print(f"\n===== {name}: {cmd}", flush=True)
    with open(f"logs/{name}.log", "w") as log:
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            log.write(line); log.flush()
            if not line.startswith(("Loading checkpoint", "Fetching")) and "it/s]" not in line:
                print(line, end="", flush=True)
        p.wait()
    st = {"returncode": p.returncode, "minutes": round((time.time() - t0) / 60, 1)}
    print(name, st, flush=True)
    return st

status = {}
status["tests"] = run("tests", "python -m pytest -q -p no:cacheprovider tests/test_llmtools.py")
# 1-2 minutes with a 0.5B model on a scratch copy: checks the whole GPU path before the long run.
shutil.rmtree("/kaggle/tmp_smoke", ignore_errors=True); os.makedirs("/kaggle/tmp_smoke", exist_ok=True)
shutil.copytree("work_full", "/kaggle/tmp_smoke/work")
status["smoke"] = run("smoke", "python scripts/llm_r2.py run --work /kaggle/tmp_smoke/work --models Qwen/Qwen2.5-0.5B-Instruct "
                      "--identify Qwen/Qwen2.5-0.5B-Instruct --test 6 --dev 2 --id-max 3 --cache /tmp/hf_smoke")
for l in open("/kaggle/tmp_smoke/work/queries/llm_multi.jsonl", encoding="utf-8").readlines()[:6] if os.path.exists("/kaggle/tmp_smoke/work/queries/llm_multi.jsonl") else []:
    print(json.loads(l)["type"], json.loads(l)["text"])
shutil.rmtree("/kaggle/tmp_smoke", ignore_errors=True)

In [ ]:
status["r2"] = run("r2", "python scripts/llm_r2.py run --work work_full --deadline-min 600")
json.dump(status, open("logs/status_r2.json", "w"), indent=1)
print(status)

In [ ]:
out = "/kaggle/working/visongsearch_results_r2.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for pat in ["work_full/queries/llm_multi*.jsonl", "work_full/results/llm_*", "work_full/runs/llm_*.trec", "logs/*"]:
        for path in glob.glob(pat):
            if os.path.isfile(path):
                z.write(path)
print(out, round(os.path.getsize(out) / 1e6, 1), "MB")
print(json.dumps(json.load(open("work_full/results/llm_multi_stats.json")), ensure_ascii=False)[:3000]
      if os.path.exists("work_full/results/llm_multi_stats.json") else "no stats")